# 05 — Final Merge, Deduplication, Outlier Handling & Standardization (Team 5)

## Responsibility
This notebook is the final stage of Phase 1. It must only be run AFTER all four
team notebooks (01–04) have been completed and verified.

## How to Use
1. Copy the complete code from notebooks 01, 02, 03, and 04 into the designated
   sections below (marked with banners).
2. Each section produces its respective df_team1 / df_team2 / df_team3 / df_team4.
3. This notebook then merges them, deduplicates, handles outliers, and standardizes.

## What is Expected as Output
- A single DataFrame with ALL columns from teams 1–4 merged column-wise
- Zero duplicate rows
- Outliers detected and handled (capping at 1.5×IQR or removal — team decides and justifies)
- All columns standardized: mean=0, std=1 using StandardScaler
- Zero null values
- Zero non-numeric columns
- Saved as processed_dataset.csv to data/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

RAW_URL = "https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/raw_dataset.csv"
df_full = pd.read_csv(RAW_URL)
print(f"Loaded dataset: {df_full.shape}")

In [ ]:
# ════════════════════════════════════════════════════════════
# PASTE TEAM 1 CODE HERE (from 01_core_numeric.ipynb)
# Expected output variable: df_team1
# ════════════════════════════════════════════════════════════

In [ ]:
# ════════════════════════════════════════════════════════════
# PASTE TEAM 2 CODE HERE (from 02_categorical_encoding.ipynb)
# Expected output variable: df_team2
# ════════════════════════════════════════════════════════════

In [ ]:
# ════════════════════════════════════════════════════════════
# PASTE TEAM 3 CODE HERE (from 03_paint_damage.ipynb)
# Expected output variable: df_team3
# ════════════════════════════════════════════════════════════

In [ ]:
# ════════════════════════════════════════════════════════════
# PASTE TEAM 4 CODE HERE (from 04_technical_specs.ipynb)
# Expected output variable: df_team4
# ════════════════════════════════════════════════════════════

In [ ]:
# Merge all four team outputs column-wise
df_final = pd.concat([df_team1, df_team2, df_team3, df_team4], axis=1)
print(f"After merge: {df_final.shape}")

## Step 1 — Remove Duplicate Rows

In [ ]:
# Drop duplicate rows and report how many were removed
n_before = len(df_final)
df_final = df_final.drop_duplicates()
n_removed = n_before - len(df_final)
print(f"Removed {n_removed} duplicate rows.")
print(f"Shape after deduplication: {df_final.shape}")

## Step 2 — Outlier Detection & Handling

In [ ]:
# Cap outliers at Q1 - 1.5*IQR and Q3 + 1.5*IQR for each numeric column
capped_summary = {}
for col in df_final.select_dtypes(include='number').columns:
    Q1 = df_final[col].quantile(0.25)
    Q3 = df_final[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_capped = ((df_final[col] < lower) | (df_final[col] > upper)).sum()
    df_final[col] = df_final[col].clip(lower=lower, upper=upper)
    if n_capped > 0:
        capped_summary[col] = n_capped

print("Values capped per column (only columns with at least 1 capped value):")
for col, n in capped_summary.items():
    print(f"  {col}: {n}")

## Step 3 — StandardScaler (mean=0, std=1)

In [ ]:
# Apply StandardScaler to all columns
scaler = StandardScaler()
df_final = pd.DataFrame(
    scaler.fit_transform(df_final),
    columns=df_final.columns,
    index=df_final.index
)
# Confirm mean ≈ 0 and std ≈ 1 for first 5 columns
print("Mean of first 5 columns (should be ≈ 0):")
print(df_final.iloc[:, :5].mean().round(6))
print("\nStd of first 5 columns (should be ≈ 1):")
print(df_final.iloc[:, :5].std().round(6))

In [ ]:
# Final sanity checks
assert df_final.isnull().sum().sum() == 0, "❌ Nulls still present!"
assert df_final.select_dtypes(exclude='number').shape[1] == 0, "❌ Non-numeric columns present!"
print(f"✅ Final dataset ready. Shape: {df_final.shape}")
print(df_final.describe().round(3))

In [ ]:
# Save output
OUTPUT_PATH = "processed_dataset.csv"
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Saved as {OUTPUT_PATH}")
print("Upload this file to: data/processed_dataset.csv in the GitHub repo.")